In [4]:
from spacerocks.observing import Observatory
from spacerocks.time import Time
from spacerocks import SpaceRock

from spacerocks.spice import SpiceKernel
kernel = SpiceKernel()
# kernel.load("/Users/kjnapier/data/spice/latest_leapseconds.tls")
kernel.load_spk("/Users/kjnapier/data/spice/de440s.bsp")
kernel.load_bpc("/Users/kjnapier/data/spice/earth_1962_240827_2124_combined.bpc")

import numpy as np

origin = "SSB"
reference_plane = "J2000"


import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

from spacerocks.nbody import Simulation, Integrator

In [5]:
w84 = Observatory.from_obscode('w84')

In [7]:
epoch = Time.now()

planets_names = ["sun", 
                 "jupiter barycenter", 
                 "saturn barycenter", 
                 "uranus barycenter", 
                 "neptune barycenter"]
rocks = [SpaceRock.from_horizons("Arrokoth", epoch=Time.now(), origin="ssb", reference_plane="J2000")]
planets = [SpaceRock.from_spice(name, epoch, reference_plane="J2000", origin='ssb', kernel=kernel) for name in planets_names]

sim = Simulation()

sim.set_epoch(epoch)
sim.set_reference_plane("J2000")
sim.set_origin('ssb')
sim.set_integrator(Integrator.ias15(timestep=20.0))


for planet in planets:
    sim.add(planet)
    
for rock in rocks:
    sim.add(rock)


In [8]:
observations = []    
for idx in range(0, 600, 5):
    sim.integrate(epoch + idx)
    observer = w84.at(epoch + idx, reference_plane=reference_plane, origin=origin, kernel=kernel)
    rock = sim.get_particle("Arrokoth")
    obs = rock.observe(observer)
    observations.append(obs)

In [9]:
from spacerocks.orbfit import gauss

In [10]:
fit_rocks = gauss(o1=observations[0], o2=observations[20], o3=observations[10], min_distance=0)
for fit_rock in fit_rocks:
    print(fit_rock.a(), fit_rock.e(), fit_rock.inc(), fit_rock.node(), fit_rock.arg())

44.23605993043258 0.04035956187985228 0.3694577226669126 0.042358330263243864 6.003774382152638
0.855600981898524 0.24129973763851323 0.4117419183454562 0.019838100355592955 0.6201715198594786
0.8820748023856904 0.29174603083253103 2.79707458656636 3.078517521454523 2.716213526254126
